# Approach 1 (Truncated BN≤290) — Averaged CE vs Fitted Curve

For every (P%, BS) combination: scatter shows averaged raw CE **up to BN=290**,
overlaid with the smooth fitted curve `A + B/(BN+1)^n` from the truncated fit.
The fitted curve extends beyond BN=290 toward BNL to show extrapolation.

Parameters come from `approach_1_truncate/intermediate/approach_1_fit_params_bs_{bs}.csv`.

PNGs saved to `BS_{bs}/fitting_avg_plot_A_1_trunc_p_{p}_bs_{bs}.png`.

In [4]:
# === Cell 1 — Config, imports ===
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ──────────────────────────────────────────────────────────────────
BN_MAX = 290   # scatter shown only up to this BN; must match compute notebook
# ─────────────────────────────────────────────────────────────────────────

BASE_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
INTER_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test\approach_1_truncate\intermediate"
OUT_DIR   = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test\approach_1_truncate\avg_plot_v_fitting_curve_truncate"

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # ln(10) ≈ 2.302585

# Auto-detect pruning levels from directory names
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning levels: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")
print(f"BN_MAX = {BN_MAX}  (scatter window)")
print("Cell 1 ready.")

Found 19 pruning levels: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
BN_MAX = 290  (scatter window)
Cell 1 ready.


In [5]:
# === Cell 2 — Load all (P%, BS) averaged data and truncated fit parameters ===
records = []   # one dict per (p, bs)

for bs in BATCH_SIZES:
    params_csv = os.path.join(INTER_DIR, f"approach_1_fit_params_bs_{bs}.csv")
    if not os.path.exists(params_csv):
        print(f"[SKIP] Missing params CSV: {params_csv}")
        continue
    params_df = pd.read_csv(params_csv)
    params_df.columns = params_df.columns.str.strip()

    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — no averaged CSV")
            continue

        row = params_df[np.isclose(params_df["P%"], p * 100)]
        if row.empty:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — no fit params row")
            continue

        avg_df = pd.read_csv(avg_csv)
        avg_df.columns = avg_df.columns.str.strip()
        ce_col = next((c for c in avg_df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in avg_df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — unexpected columns {list(avg_df.columns)}")
            continue
        avg_df = avg_df.dropna(subset=[ce_col, bn_col])
        avg_df = avg_df[avg_df[bn_col] <= BN_MAX]   # show only truncated range

        learn_BN           = float(row["learn_BN"].iloc[0])
        avg_CE_learn_at_BN = float(row["avg_CE_learn_at_BN"].iloc[0]) \
                             if "avg_CE_learn_at_BN" in row.columns else np.nan
        from_data = np.isfinite(avg_CE_learn_at_BN)

        records.append({
            "bs": bs, "p": p,
            "bn_avg":             avg_df[bn_col].values.astype(float),
            "ce_avg":             avg_df[ce_col].values.astype(float),
            "A":                  float(row["A"].iloc[0]),
            "B":                  float(row["B"].iloc[0]),
            "n":                  float(row["n"].iloc[0]),
            "CE_L":               float(row["CE_L"].iloc[0]),
            "learn_BN":           learn_BN,
            "avg_CE_learn_at_BN": avg_CE_learn_at_BN,
            "from_data":          from_data,
            "IPA":                float(row["IPA"].iloc[0]),
        })
        ce_str = f"{avg_CE_learn_at_BN:.4f}" if from_data else "NaN"
        src    = "data" if from_data else "analytic fallback"
        print(f"  OK   P%={p*100:5.1f}%  BS={bs:>6}  "
              f"learn_BN={learn_BN!r}  avg_CE@BNL={ce_str}  [{src}]")

print(f"\nLoaded {len(records)} combinations.")

  OK   P%=  0.0%  BS=    64  learn_BN=41.0  avg_CE@BNL=0.5003  [data]
  OK   P%= 10.0%  BS=    64  learn_BN=47.0  avg_CE@BNL=0.4912  [data]
  OK   P%= 20.0%  BS=    64  learn_BN=52.0  avg_CE@BNL=0.4897  [data]
  OK   P%= 30.0%  BS=    64  learn_BN=58.0  avg_CE@BNL=0.4888  [data]
  OK   P%= 40.0%  BS=    64  learn_BN=49.0  avg_CE@BNL=0.5472  [data]
  OK   P%= 50.0%  BS=    64  learn_BN=86.0  avg_CE@BNL=0.4742  [data]
  OK   P%= 60.0%  BS=    64  learn_BN=85.0  avg_CE@BNL=0.5216  [data]
  OK   P%= 70.0%  BS=    64  learn_BN=160.0  avg_CE@BNL=0.4741  [data]
  OK   P%= 80.0%  BS=    64  learn_BN=304.0  avg_CE@BNL=NaN  [analytic fallback]
  OK   P%= 82.0%  BS=    64  learn_BN=368.0  avg_CE@BNL=NaN  [analytic fallback]
  OK   P%= 84.0%  BS=    64  learn_BN=463.0  avg_CE@BNL=NaN  [analytic fallback]
  OK   P%= 86.0%  BS=    64  learn_BN=617.0  avg_CE@BNL=NaN  [analytic fallback]
  OK   P%= 88.0%  BS=    64  learn_BN=897.0  avg_CE@BNL=NaN  [analytic fallback]
  OK   P%= 90.0%  BS=    64  learn

In [6]:
# === Cell 3 — Plot averaged CE (BN≤290) vs fitted curve for every (P%, BS) ===
plt.rcParams.update({"font.size": 13})

for rec in records:
    bs                 = rec["bs"]
    p                  = rec["p"]
    bn_avg             = rec["bn_avg"]
    ce_avg             = rec["ce_avg"]
    A                  = rec["A"]
    B                  = rec["B"]
    n                  = rec["n"]
    CE_L               = rec["CE_L"]
    learn_BN           = rec["learn_BN"]
    avg_CE_learn_at_BN = rec["avg_CE_learn_at_BN"]
    from_data          = rec["from_data"]
    IPA                = rec["IPA"]

    # x-axis extends to cover BNL even when it's beyond BN_MAX
    bn_end    = max(bn_avg.max() if len(bn_avg) > 0 else BN_MAX,
                    learn_BN if np.isfinite(learn_BN) else 0) * 1.3
    bn_smooth = np.linspace(0, bn_end, 600)
    y_fit     = A + B / ((bn_smooth + 1) ** n)

    fig, ax = plt.subplots(figsize=(10, 6))

    # Averaged CE scatter (truncated to BN≤290)
    ax.scatter(bn_avg, ce_avg, s=8, color="#aaaaaa", alpha=0.6, zorder=1,
               label=r"$\overline{CE}_{test}$ (avg of 100 runs, BN\u2264290)")

    # Smooth fitted curve (extends beyond BN_MAX to show extrapolation)
    ax.plot(bn_smooth, y_fit, color="#1f77b4", linewidth=2.2, zorder=3,
            label=f"Fit (BN\u2264290): A={A:.4f},  B={B:.4f},  n={n:.4f}")

    # BN_MAX boundary line
    ax.axvline(BN_MAX, color="#bbbbbb", linewidth=1.0, linestyle="-", alpha=0.6)
    ax.text(BN_MAX + 5, CE_o - 0.05, f"BN={BN_MAX}\n(fit cutoff)",
            fontsize=8, color="#888888", va="top")

    # Horizontal reference lines
    ax.axhline(CE_o, color="#888888", linewidth=1.0, linestyle=":")
    ax.text(bn_end, CE_o + 0.03, f"CE_o = {CE_o:.4f}",
            ha="right", fontsize=10, color="#666666")

    ax.axhline(CE_L, color="#9467bd", linewidth=1.4, linestyle="--")
    ax.text(bn_end, CE_L + 0.03, f"CE_L = {CE_L:.4f}",
            ha="right", fontsize=10, color="#9467bd")

    ax.axhline(A, color="#2ca02c", linewidth=1.4, linestyle="--")
    ax.text(bn_end, A - 0.07, f"A = {A:.4f}  (asymptote)",
            ha="right", fontsize=10, color="#2ca02c")

    # BNL marker
    if np.isfinite(learn_BN) and learn_BN > 0:
        if from_data:
            bnl_color = "#d62728"
            ax.axvline(learn_BN, color=bnl_color, linewidth=1.4,
                       linestyle="--", alpha=0.8, zorder=4)
            ax.scatter([learn_BN], [avg_CE_learn_at_BN], s=80, color=bnl_color,
                       marker="*", zorder=5, label=f"Avg data crossing  (BN={learn_BN:.0f})")
            annot_text = f"BNL = {learn_BN:.0f}  [avg data]\nIPA = {IPA:.5f}"
        else:
            bnl_color = "#ff7f0e"
            ax.axvline(learn_BN, color=bnl_color, linewidth=1.4,
                       linestyle=":", alpha=0.8, zorder=4)
            annot_text = f"BNL = {learn_BN:.0f}  [extrapolated]\nIPA = {IPA:.5f}"

        ax.annotate(
            annot_text,
            xy=(learn_BN, CE_L),
            xytext=(learn_BN + bn_end * 0.03, CE_L + 0.15),
            fontsize=10, color=bnl_color,
            arrowprops=dict(arrowstyle="->", color=bnl_color, lw=1.0)
        )

    ax.set_xlabel("Batch Number (BN)")
    ax.set_ylabel("CE_TEST")
    ax.set_xlim(0, bn_end)
    ax.set_ylim(max(0, A - 0.15), CE_o + 0.25)
    src_tag = "avg data" if from_data else "extrapolated"
    ax.set_title(
        f"Approach 1 (BN\u2264290 truncated) — Avg CE vs Fit  |  P%={p*100:.1f}%  BS={bs}\n"
        f"BNL source: {src_tag}"
    )
    ax.legend(fontsize=10, frameon=False, loc="upper right")
    ax.grid(True, alpha=0.25)

    bs_dir  = os.path.join(OUT_DIR, f"BS_{bs}")
    os.makedirs(bs_dir, exist_ok=True)
    out_png = os.path.join(bs_dir, f"fitting_avg_plot_A_1_trunc_p_{p}_bs_{bs}.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: fitting_avg_plot_A_1_trunc_p_{p}_bs_{bs}.png  [{src_tag}]")

print("\n[Done]")

  Saved: fitting_avg_plot_A_1_trunc_p_0.0_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.1_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.2_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.3_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.4_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.5_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.6_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.7_bs_64.png  [avg data]
  Saved: fitting_avg_plot_A_1_trunc_p_0.8_bs_64.png  [extrapolated]
  Saved: fitting_avg_plot_A_1_trunc_p_0.82_bs_64.png  [extrapolated]
  Saved: fitting_avg_plot_A_1_trunc_p_0.84_bs_64.png  [extrapolated]
  Saved: fitting_avg_plot_A_1_trunc_p_0.86_bs_64.png  [extrapolated]
  Saved: fitting_avg_plot_A_1_trunc_p_0.88_bs_64.png  [extrapolated]
  Saved: fitting_avg_plot_A_1_trunc_p_0.9_bs_64.png  [extrapolated]
  Saved: fitting_avg_plot_A_1_trunc_p_0.92_bs_64.png  [avg data]
  Saved: fi